## Chargement des Données via SQLAlchemy

Configurer la connexion SQLAlchemy

In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

if not all([user, password, db]):
    raise ValueError("One or more DB environment variables are missing! Check your .env file.")

# connection string
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db}")

# test connection
try:
    with engine.connect() as conn:
        print('Connection successful!')
except Exception as e:
    print(f'connection failed: {e}')

Infrastructure: Automated Database Setup

In [ ]:

def run_sql_file(filename, connection):
    from sqlalchemy import text
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            sql_command = f.read()
        with connection.begin() as conn:
            conn.execute(text(sql_command))
        print(f"✅ {filename} executed successfully!")
    except FileNotFoundError:
        print(f"❌ Error: {filename} not found. Check your file path!")

# building the entire database structure in order
scripts = [
    "../sql/drop_tables.sql",
    "../sql/create_tables.sql", 
    "../sql/add_constraints.sql", 
    "../sql/performance.sql"
]
for script in scripts:
    run_sql_file(script, engine)

Séparer financecore_clean.csv selon les tables normalisées avant insertion


In [ ]:
import pandas as pd
df = pd.read_csv("../data/financecore_clean.csv")

print(df.head())
print(df.columns)

In [ ]:
df['date_transaction'] = pd.to_datetime(df['date_transaction'])

# Dimensions
agencies_df = df[['agence']].drop_duplicates().reset_index(drop=True)
products_df = df[['produit']].drop_duplicates().reset_index(drop=True)
categories_df = df[['categorie']].drop_duplicates().reset_index(drop=True)
clients_df = df[['client_id', 'score_credit_client', 'categorie_risque', 'segment_client']].drop_duplicates(subset=['client_id'])

# Time Dimension
dim_date = pd.DataFrame({'date_transaction': df['date_transaction'].unique()})
dim_date['annee'] = dim_date['date_transaction'].dt.year
dim_date['mois'] = dim_date['date_transaction'].dt.month
dim_date['trimestre'] = dim_date['date_transaction'].dt.quarter
dim_date['jour_semaine'] = dim_date['date_transaction'].dt.day_name()

print("Step 'Séparer financecore_clean.csv' is complete!")
print(f'- {len(agencies_df)} Unique agencies')
print(f'- {len(products_df)} Unique products')
print(f'- {len(categories_df)} Unique categories')
print(f'- {len(clients_df)} Unique clients')

Insérer les données

In [ ]:
# A. Safe insertion function
def safe_insert(dataframe, table_name, engine):
    try:
        dataframe.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"✅ {table_name}: Inserted.")
    except Exception:
        print(f"ℹ️ {table_name}: Already contains data, skipping.")

# B. Insert Dimensions
safe_insert(agencies_df, 'agencies', engine)
safe_insert(products_df, 'products', engine)
safe_insert(categories_df, 'categories', engine)
safe_insert(clients_df, 'clients', engine)
safe_insert(dim_date, 'dim_date', engine)

# C. Map IDs for the Transactions Table
agencies_db = pd.read_sql("SELECT agence_id, agence FROM agencies", engine)
products_db = pd.read_sql("SELECT produit_id, produit FROM products", engine)
categories_db = pd.read_sql("SELECT categorie_id, categorie FROM categories", engine)
dates_db = pd.read_sql("SELECT date_id, date_transaction FROM dim_date", engine)

# D. Merge logic
df_mapped = df.merge(agencies_db, on='agence') \
              .merge(products_db, on='produit') \
              .merge(categories_db, on='categorie') \
              .merge(dates_db, on='date_transaction')

# E. Select exactly the 14 columns defined in your 'create_tables.sql'
transactions_df = df_mapped[[
    'transaction_id', 'client_id', 'agence_id', 'produit_id', 'categorie_id', 
    'date_id', 'montant', 'type_operation', 'devise', 'taux_change_eur', 
    'montant_eur_verifie', 'statut', 'is_anomaly', 'solde_avant'
]]

# F. Final Insertion
try:
    transactions_df.to_sql('transactions', engine, if_exists='append', index=False)
    print(f"🚀 SUCCESS: {len(transactions_df)} transactions inserted!")
except Exception as e:
    print(f"❌ Final insertion failed: {e}")

Implémenter la gestion des conflits (ON CONFLICT DO NOTHING / DO UPDATE)

In [ ]:
from sqlalchemy import text

# 1. DE-DUPLICATE in Python first
# This keeps only the LATEST version of each transaction ID in your current batch
clean_batch_df = transactions_df.drop_duplicates(subset=['transaction_id'], keep='last')

# 2. Upload the clean batch to a temporary table
clean_batch_df.to_sql('temp_transactions', engine, if_exists='replace', index=False)

# 3. Execute the UPSERT
upsert_sql = text("""
INSERT INTO transactions (
    transaction_id, client_id, agence_id, produit_id, categorie_id, 
    date_id, montant, type_operation, devise, taux_change_eur, 
    montant_eur_verifie, statut, is_anomaly, solde_avant
)
SELECT * FROM temp_transactions
ON CONFLICT (transaction_id) 
DO UPDATE SET 
    statut = EXCLUDED.statut,
    is_anomaly = EXCLUDED.is_anomaly,
    solde_avant = EXCLUDED.solde_avant,
    montant = EXCLUDED.montant,
    montant_eur_verifie = EXCLUDED.montant_eur_verifie;

-- Clean up
DROP TABLE temp_transactions;
""")

with engine.begin() as conn:
    conn.execute(upsert_sql)
    print(f"✅ Successfully processed {len(clean_batch_df)} unique transactions.")

Vérifier l'intégrité référentielle après chargement (COUNT…..)

In [ ]:
# 1. Ensure Python count only includes UNIQUE transaction IDs
final_unique_df = transactions_df.drop_duplicates(subset=['transaction_id'])

# 2. Re-run the Integrity Check
csv_count = len(final_unique_df)
db_count = pd.read_sql("SELECT COUNT(*) FROM transactions", engine).iloc[0,0]

print("--- Final Sync Report ---")
print(f"Unique Transactions (Python): {csv_count}")
print(f"Total Transactions (Database): {db_count}")

if db_count == csv_count:
    print("✅ Perfect Sync: Database matches unique CSV records.")
else:
    print(f"ℹ️ Total records in DB: {db_count}. Your current batch has {csv_count} unique rows.")

## Requêtes SQL Analytiques Avancées

Agrégations : total et moyenne des transactions par agence, produit et mois (GROUP BY, HAVING)

In [ ]:
# Query the performance view created in performance.sql
query_performance = "SELECT * FROM vw_analytics_performances;"
df_performance = pd.read_sql(query_performance, engine)

# Display the top performers
df_performance.sort_values(by='volume_total_eur', ascending=False).head(10)

Sous-requêtes : clients avec un solde inférieur à la moyenne nationale

In [ ]:
# Query the view using the name defined in performance.sql
query_low_balance = "SELECT * FROM vw_clients_sous_moyenne;"
df_low_balance = pd.read_sql(query_low_balance, engine)
df_low_balance.head()

CASE WHEN : calcul du taux de défaut par segment de risque

In [ ]:
query_risk = "SELECT * FROM vw_taux_defaut_segment;"
df_risk = pd.read_sql(query_risk, engine)
df_risk

Jointures multi-tables

In [ ]:
# Query the master view that joins: transactions, clients, agencies, products, categories, and dim_date
query_joins = "SELECT * FROM vw_transactions_full LIMIT 10;"
df_joins = pd.read_sql(query_joins, engine)

# Display the result to prove all tables are connected
df_joins

Créer des vues analytiques pour les KPIs du dashboard (brief 3)

In [ ]:
query_kpis = "SELECT * FROM vw_dashboard_kpis;"
df_kpis = pd.read_sql(query_kpis, engine)
df_kpis